<a href="https://colab.research.google.com/github/NaveenHerath/llm-from-scratch/blob/main/day6_attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A practice Tensor
Understanding how transormers works.

In [11]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [12]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)
print(x.shape)
print(x[0]) #print the 0th index chunk of 4 chunks
#an exmple tensor which is a random 4,8,32 where 4 chunks of 8 characters each with 2 weights

torch.Size([4, 8, 2])
tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])


In [13]:
# version1 #getting the average of preceding character weights using a for loop
xbow  = torch.zeros((B, T, C))
for b in range(B):
  for t in range(T):
    xprev = x[b, :t+1]
    xbow[b, t] = torch.mean(xprev, 0)
print(xbow[0])

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])


In [14]:
#version2 #getting the average using matrix multiplication.
#a toy ex
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0, 10, (3, 2)).float()
c = a @ b
print(a)
print(b)
print(c)

tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [15]:
#version 2
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x      # (T, T) @ (B, T, C) gives (B, T, C)
print(wei)
print(xbow2.shape)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])
torch.Size([4, 8, 2])


In [16]:
#version 3 - crate the wei matrix using masking and softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
print(wei)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])


In [17]:
print(torch.allclose(xbow, xbow2))
print(torch.allclose(xbow, xbow3))
print(torch.allclose(xbow2, xbow3))
print((xbow - xbow2).abs().max().item())

False
False
True
3.236345946788788e-08


In [18]:
print(torch.allclose(xbow, xbow2, atol=1e-6))
print(torch.allclose(xbow, xbow3, atol=1e-6))
print(torch.allclose(xbow2, xbow3, atol=1e-6))

True
True
True


All three versions give identical results.

Now we have  a new matrix called xbow where we can store, the information about the previous characters.

We made that **by calculating the weighted sum for each position only looking at the perceeding characters**.

MATRIX MULTIPLICATION

The previous weighted sum method gets the plain average.  Basically it gives the same weight for all the perceeding characters. So to avoid that we introduce **QUERY, KEY & VALUE**

In [19]:
torch.manual_seed(1337)
B, T, C = 4, 8, 32     # 32 channels now, like n_embd on Day 3
x = torch.randn(B, T, C)

head_size = 16
key   = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)      # (B, T, 16)
q = query(x)    # (B, T, 16)
wei = q @ k.transpose(-2, -1)    # (B, T, 16) @ (B, 16, T) gives (B, T, T)

tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)    # (B, T, 16)
out = wei @ v   # (B, T, T) @ (B, T, 16) gives (B, T, 16)

print(out.shape)

torch.Size([4, 8, 16])


In [20]:
print(wei[0, -1])
print(wei[0, -1].argmax().item())

tensor([0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391],
       grad_fn=<SelectBackward0>)
6


In [21]:
n_embd = 32       # numbers describing each position
block_size = 8    # the longest chunk the model will see
head_size = 16

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)                    # (B, T, head_size)
        q = self.query(x)                  # (B, T, head_size)
        wei = q @ k.transpose(-2, -1)      # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)       # (B, T, T)
        v = self.value(x)                  # (B, T, head_size)
        out = wei @ v                      # (B, T, head_size)
        return out

In [22]:
torch.manual_seed(1337)
x = torch.randn(4, 8, n_embd)
head = Head(head_size)
out_head = head(x)
print(out_head.shape)
print(torch.equal(out_head, out))
print(sum(p.numel() for p in head.parameters()))

torch.Size([4, 8, 16])
True
1536


In [23]:
x2 = x.clone()
x2[:, 7] = torch.randn(4, n_embd)     # new numbers at the last position
out2 = head(x2)
print(torch.equal(out_head[:, :7], out2[:, :7]))   # positions 0 to 6 the same?
print(torch.equal(out_head[:, 7], out2[:, 7]))     # position 7 the same?

True
False
